# WISPR Cloud demo
May 2026

This is a very simple demo that (a) looks up all availble WISRP L2 or L3 files then (b) reads them in and (c) calculates a difference image on only the Inner or Outer (user choice) files.  It is a trivial example, but shows how to rip through a lot of data quickly.  Using [Encounter 9](https://wispr.nrl.navy.mil/encounter-summaries) as an example, it takes ~4-5 minutes to load and compute on all the ~800 FITS files in a given Encounter.  This uses a single CPU and reads in files in order. Faster threaded loading should be added for real work, which you can model after the PSP EPILo threading examples, for future work. And yes, difference images aren't per se revelant for WISPR but this code does show how to process WISPR cloud data.

For now, this serves as a demo on how to access WISPR cloud files. For this demo, it prompts the user on dataset (L2 or L3, Inner or Outer), whether to do a quick test run or a full rip through all the data, and how often to do incremental plots.  The defaults are for a quick L2 Inner test run.

In [ ]:
import re
import time
import cloudcatalog as cc
import matplotlib.pyplot as plt
import matplotlib.colors as colors
from matplotlib.colors import LogNorm
import astropy
import astropy.units as u
from astropy.io import fits
from astropy.visualization import ImageNormalize, SqrtStretch
import numpy as np
import sunpy.map
from IPython.display import display

# helper routines for all tasks

def fetch_wispr():
    """ Front end for CloudCatalog, including prompting user to choose L2 or L3 data """
    query = input("\n> L2 (default) or L3 data? (2/3): ")
    if '3' in query:
        dataid = "PARKERSOLARPROBE_WISPR_FITS_LEVEL2_PT30M"
    else:
        dataid = "PARKERSOLARPROBE_WISPR_FITS_LEVEL3_PT30M"

    # always initialize CloudCatalog before querying!
    fr = cc.CloudCatalog("s3://gov-nasa-hdrl-data1/")

    # displaying and choosing time ranges
    meta = fr.get_entry(dataid)
    print(f"Metadata for dataset {dataid}: {meta}")
    #start, stop = "2022-05-27T00:00:00Z", "2022-06-07T23:59:59Z" # Encounter 12
    #start, stop = "2022-09-01T00:00:00Z", "2022-09-11T23:59:59Z" # Encounter 13
    start, stop = "2021-08-04T00:00:00Z", "2021-08-15T23:59:59Z" # Encounter 9
    print(f"Fetching filelists for {dataid}...")

    # the actual CloudCatalog query in a one-liner
    filekeys = fr.request_cloud_catalog(dataid,start_date=start,stop_date=stop)

    return filekeys
    
def load_wispr_sunpy_fixheader(fname):
    """ Uses AstroPy to fetch cloud file, and fix non-standard FITS keywords
        that otherwise throw warnings in SunPy.
    """
    data, header = fits.getdata(fname, header=True, ignore_blank=True)
    # MSB is commonly "mean solar brightness", but not a FITS unit.
    header["BUNIT"] = "1"          # dimensionless, FITS-compatible
    header["BUNIT0"] = "MSB"       # optional: preserve original meaning
    header.pop("BLANK", None)
    return sunpy.map.Map(data, header)

def load_wispr_sunpy(fname):
    """ Alternately the standard SunPy loading routine to fetch cloud files.
        You can ignore the FITS header warnings it throws as PSP uses
        two non-standard FITS keys (BUNITS = 'MSB' and keyword 'BLANK')
    """
    smap = sunpy.map.Map(fname, fsspec_kwargs={"anon": True})
    data = np.asarray(smap.data, dtype=np.float64)
    map_mask = getattr(smap, "mask", None)
    if map_mask is not None:
        data = data.copy()
        data[np.asarray(map_mask, dtype=bool)] = fill_value
    data = np.nan_to_num(data, nan=fill_value, posinf=fill_value, neginf=fill_value)
    ys, xs = _safe_crop_slices(data.shape, crop=None)
    img = data[ys, xs]
    return img, smap, fname

def plot_wispr(smap):
    """ Plot a SunPy Map with matplotlib """
    fig = plt.figure()
    ax = fig.add_subplot(projection=smap)
    smap.plot(axes=ax,norm=ImageNormalize(vmin=0, vmax=5e3, stretch=SqrtStretch()))
    smap.draw_limb(axes=ax)
    smap.draw_grid(axes=ax)
    plt.show()

def get_norm(map1):
    """ Create a good value for doing log image plots """
    data1 = map1.data
    data1 = data1[np.isfinite(data1)]
    data1 = data1[data1 > 0]
    vmin1, vmax1 = data1[data1 > 0].min(), data1.max()
    norm1 = LogNorm(vmin=vmin1, vmax=vmax1)
    return norm1

def plot_two_maps(map1, map2, log=True, is_diff=True):
    """ Plots 2 SunPy Maps, optionally log scale.
        If is_diff is set, relabels 2nd image as 'Difference Image' """
    if log:
        norm1, norm2 = get_norm(map1), get_norm(map2)
    else:
        norm1, norm2 = None, None
    fig = plt.figure(figsize=(12, 5))
    ax1 = fig.add_subplot(1, 2, 1, projection=map1)
    im1 = map1.plot(axes=ax1, norm=norm1)
    ax1.set_title(map1.name)
    ax2 = fig.add_subplot(1, 2, 2, projection=map2)
    im2 = map2.plot(axes=ax2, norm=norm2)
    if is_diff:
        ax2.set_title('Difference Image')
    else:
        ax2.set_title(map2.name)
    plt.colorbar(im1, ax=ax1, fraction=0.046, pad=0.04)
    plt.colorbar(im2, ax=ax2, fraction=0.046, pad=0.04)
    plt.tight_layout()
    plt.show()

def filter_filelist(filtertype,filelist):
    if filtertype == 'Outer':
        rx = re.compile(r'_2\d{3}\.fits$')
    else:
        rx = re.compile(r'_1\d{3}\.fits$')
    files_xxxx = [f for f in filelist if rx.search(f)]
    return files_xxxx

def demo_user_inputs(fsize):
    """ Query user for run parameters: Inner/Outer, quick/big run, Plot every nth file """
    query = input("\n> Inner (default) or Outer images? (i/o): ")
    if query.startswith('o'):
        filtertype = "Outer" # "Outer" or "Inner"
    else:
        filtertype = "Inner"
    query = input("> Choose either 1: quick test on 10 files (default) "
                  "or 2: big test on all files. (1/2): ")
    if query.startswith('2'):
        runsize = fsize
        printcheck = int(runsize/15)
    else:
        runsize = 10
        printcheck = 5
    query = input(f"> Plot every nth file, default every {printcheck}? (#): ")
    try:
        query = int(query)
        printcheck = min(query, runsize)
    except:
        pass
    return filtertype, runsize, printcheck
 

In [ ]:
""" Main routine, prompting user, fetching file list, fetching & processing files, displaying """

# show all WISPR holdings, for fun
mysearch = cc.EntireCatalogSearch()
ids = mysearch.search_by_id('WISPR')
print(f"All WISPR data in cloud (with metadata):")
for item in ids: print(f"\tID: {item['id']}, Title: {item['title']}, Filetype: {item['filetype']}")
    
# get actual chosen WISRP filelist + user parameters
filekeys = fetch_wispr()
filtertype, runsize, printcheck = demo_user_inputs(len(filekeys))
wispr_files = filekeys['datakey'].to_list() # also extract just filenames
wispr_files = filter_filelist(filtertype, wispr_files)
runsize = min(runsize,len(wispr_files)) # post-filter sanity check
print(f"Using {filtertype}, {runsize} files, plotting every {printcheck}th\n")

# prefetch the first file that matches the user requirements
index = 0
while index < len(wispr_files):
    smap_prev = load_wispr_sunpy_fixheader(wispr_files[index])
    index += 1
    if filtertype in smap_prev.name:
        display(smap_prev)
        break
    else:
        print("ignoring",wispr_files[index])
        
# now loop through all the files
now = time.time()
max_mean, ivalid, icount = 0, 0, index
for fname in wispr_files[index:runsize]:
    smap = load_wispr_sunpy_fixheader(fname)
    ++icount
    if filtertype in smap.name:

        # actual calculations, here generating a difference image
        smap_diff = sunpy.map.Map([smap - smap_prev.quantity])
        smap_prev = smap
        # very simple demo 'processing', just getting the data mean value
        mean_val = np.nanmean(smap.data)
        if mean_val > max_mean: max_mean = mean_val

        # useful info to tally and display to user
        ivalid += 1
    if icount % printcheck == 0:
        print(f"Loaded file {icount}/{runsize}, "
              f"mean value is {mean_val} vs running highest mean {max_mean}")
        plot_two_maps(smap, smap_diff, log=True, is_diff=True)
    icount += 1
    
print("\nResults:\nHighest computed mean was ",max_mean)
print(f"Runtime for loading {runsize} files "
      f"and making {ivalid} {filtertype} diff imgs "
      f"was {int(time.time()-now)} seconds")